# Tahap 2 - Case Representation

Notebook ini mengubah putusan yang sudah bersih (hasil Tahap 1) menjadi data terstruktur sesuai siklus CBR.

**Input:** `data/raw/case_*.txt`

**Output:**
- `data/processed/cases.csv`
- `data/processed/cases.json`
- `logs/domain_validation.csv` (catatan validasi relevansi domain Perdata Waris)

Kolom utama mengikuti instruksi tugas: `case_id`, `no_perkara`, `tanggal`, `jenis_perkara`, `ringkasan_fakta`, `argumen_hukum_utama`, `pasal`, `pihak`, `text_full`. Kolom tambahan (`solution_label`, `length_jumlah_kata`, `bag_of_words`, `qa_pairs`) ditambahkan untuk mendukung tahap retrieval, reuse, dan evaluasi.

Catatan perbaikan dari versi sebelumnya:
1. Pembersihan noise disclaimer Direktori MA RI diperluas, karena sebelumnya ada potongan disclaimer yang masih lolos ke kolom `argumen_hukum_utama`.
2. Ekstraksi `argumen_hukum_utama` sekarang difokuskan pada blok diktum MENGADILI yang sesungguhnya, bukan sekadar potongan teks pertama yang mengandung kata kunci.
3. Ditambahkan validator domain otomatis untuk menandai dokumen yang kemungkinan besar bukan perkara waris, supaya dapat diperiksa dan dikeluarkan dari case base sebelum tahap retrieval.

In [1]:
from pathlib import Path
import re
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_TXT_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
LOG_DIR = PROJECT_ROOT / 'logs'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

DOMAIN_PERKARA = 'Perdata Waris'
TARGET_DOCS = 30

print('Input raw:', RAW_TXT_DIR)
print('Output processed:', PROCESSED_DIR)

Input raw: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\raw
Output processed: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\processed


In [2]:
# Noise tambahan yang sebelumnya tidak tertangkap oleh Tahap 1, karena variasi kalimat
# disclaimer Direktori MA RI cukup beragam antar dokumen (mis. "inakurasi" vs "inkompatibilitas").
# Pembersihan kedua ini dilakukan di sini agar kolom ringkasan_fakta dan argumen_hukum_utama
# tidak ikut memuat potongan disclaimer.
EXTRA_NOISE_PATTERNS = [
    r'kepaniteraan\s+berusaha\s+untuk\s+selalu\s+mencantumkan.*?(?:waktu\s+kewaktu\.|ext\.\s*\d+\))',
    r'dalam\s+hal\s+anda\s+menemukan\s+ina?kurasi.*?(?:ext\.\s*\d+\))',
    r'email\s*:\s*kepaniteraan\s*mahkamahagung\.go\.id.*?(?:ext\.\s*\d+\))',
    r'telp\s*:\s*021[\-\s]?\d+.*?(?:ext\.\s*\d+\))',
]


def strip_extra_noise(text: str) -> str:
    cleaned = text
    for pattern in EXTRA_NOISE_PATTERNS:
        cleaned = re.sub(pattern, ' ', cleaned, flags=re.IGNORECASE | re.DOTALL)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned


print('Fungsi strip_extra_noise() siap digunakan.')

Fungsi strip_extra_noise() siap digunakan.


In [24]:
BULAN = 'januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember'


def normalize_space(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text)).strip()


def first_regex(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            return normalize_space(match.group(1))
    return 'TIDAK DITEMUKAN'


def extract_no_perkara(text: str) -> str:
    patterns = [
        r'(?:nomor|no\.?|perkara\s+nomor)\s*[:\-]?\s*([0-9]{1,6}\s*(?:k|pk|pdt\.g|pdt|pid\.sus|pid|dt)?\s*/\s*[a-z0-9\.\-]+\s*/\s*[0-9]{4})',
        r'([0-9]{1,6}\s*(?:k|pk|pdt\.g|pdt|pid\.sus|pid|dt)\s*/\s*[a-z0-9\.\-]+\s*/\s*[0-9]{4})',
    ]
    return first_regex(patterns, text).upper()


def extract_tanggal(text: str) -> str:
    patterns = [
        rf'(?:tanggal|pada\s+tanggal|diputuskan\s+pada\s+tanggal|diucapkan\s+pada\s+tanggal)\s+(\d{{1,2}}\s+(?:{BULAN})\s+\d{{4}})',
        rf'(\d{{1,2}}\s+(?:{BULAN})\s+\d{{4}})',
    ]
    return first_regex(patterns, text).lower()


def extract_pasal(text: str) -> str:
    # Buat teks versi super rapat tanpa spasi sama sekali khusus untuk deteksi kata kunci
    text_raw = str(text).lower()
    text_condensed = re.sub(r'\s+', '', text_raw)
    
    cleaned = []

    # 1. Cari pola pasal menggunakan teks super rapat (contoh: "pasal832" atau "pasal1365")
    # Pola ini akan menangkap p a s a l maupun pasal biasa
    pola_pasal_rapat = r'pasal(\d+)'
    pasal_matches = re.findall(pola_pasal_rapat, text_condensed)
    
    for num in pasal_matches:
        # Cek undang-undang terdekat di teks asli
        label_pasal = f"PASAL {num}"
        if f"pasal{num}kuh" in text_condensed or "kuhperdata" in text_condensed or "bw" in text_condensed:
            label_pasal += " KUHPERDATA"
        elif f"pasal{num}khi" in text_condensed or "khi" in text_condensed:
            label_pasal += " KHI"
        
        if label_pasal not in cleaned:
            cleaned.append(label_pasal)

    if cleaned:
        return ', '.join(cleaned[:10])

    # 2. Ambil FALLBACK dengan mendeteksi kata kunci substansial (versi rapat)
    # Ini mendeteksi "hukumadat", "adat", "khi", "kuhperdata", "hukumislam", dll.
    fallback_map = {
        'adat': 'HUKUM ADAT',
        'khi': 'KOMPILASI HUKUM ISLAM (KHI)',
        'kuhper': 'KUHPERDATA',
        'burgelijk': 'KUHPERDATA / BW',
        'faraid': 'HUKUM ISLAM / FARAID',
        'syariah': 'HUKUM ISLAM',
        'islam': 'HUKUM ISLAM',
        'boedel': 'HUKUM WARIS / BOEDEL',
        'hibah': 'HUKUM WARIS / HIBAH',
        'wasiat': 'HUKUM WARIS / WASIAT',
        'waris': 'HUKUM WARIS' # Jika tidak ada indikator spesifik, cap sebagai hukum waris umum
    }

    fallback = []
    for keyword, label in fallback_map.items():
        if keyword in text_condensed:
            fallback.append(label)

    # Hilangkan duplikat hukum nasional/adat/islam yang kontradiktif jika diperlukan, 
    # tapi sementara kita keluarkan semua yang relevan agar tabel isi tidak kosong
    fallback = sorted(set(fallback))
    
    return ', '.join(fallback[:3]) if fallback else 'TIDAK DITEMUKAN'


def extract_pihak(text: str) -> str:
    t = normalize_space(text.lower())

    match = re.search(
        r'antara\s+(.{5,300}?)\s+(?:melawan|lawan)\s+(.{5,300}?)(?:duduk perkara|menimbang|membaca|pengadilan|tentang|$)',
        t,
        flags=re.IGNORECASE
    )
    if match:
        pihak_1 = normalize_space(match.group(1))[:120]
        pihak_2 = normalize_space(match.group(2))[:120]
        return f'{pihak_1.upper()} VS {pihak_2.upper()}'

    party_keywords = [
        'penggugat', 'para penggugat',
        'tergugat', 'para tergugat',
        'pemohon', 'para pemohon',
        'termohon', 'para termohon',
        'pembanding', 'terbanding',
        'pemohon kasasi', 'termohon kasasi',
        'penggugat asal', 'tergugat asal',
        'ahli waris'
    ]

    found = []
    for kw in party_keywords:
        if re.search(rf'\b{re.escape(kw)}\b', t, flags=re.IGNORECASE):
            found.append(kw.upper())

    if found:
        return ' / '.join(sorted(set(found)))

    if re.search(r'\bpenggugat\b|\btergugat\b|\bpemohon\b|\btermohon\b|\bahli waris\b', t):
        return 'PIHAK PERDATA TERDETEKSI, NAMA TIDAK TERSTRUKTUR'

    return 'TIDAK DITEMUKAN'


def extract_section(text: str, start_keywords, end_keywords=None, max_words=350) -> str:
    text_lc = text.lower()
    start_idx = -1

    for kw in start_keywords:
        idx = text_lc.find(kw)
        if idx != -1:
            start_idx = idx
            break

    if start_idx == -1:
        words = text.split()[:max_words]
        return ' '.join(words)

    end_idx = len(text)
    if end_keywords:
        for kw in end_keywords:
            idx = text_lc.find(kw, start_idx + 10)
            if idx != -1:
                end_idx = min(end_idx, idx)

    section = text[start_idx:end_idx]
    words = section.split()[:max_words]
    return ' '.join(words)


def extract_amar_putusan(text: str, max_words=350) -> str:
    """
    Ekstraksi argumen_hukum_utama / amar putusan.

    Perbaikan dari versi sebelumnya: pencarian sekarang difokuskan pada blok
    diktum MENGADILI yang sesungguhnya (ditandai kata kapital terpisah spasi
    seperti "M E N G A D I L I" atau kata "mengadili" yang diikuti pola amar
    seperti "dalam pokok perkara", "mengabulkan", "menolak"), bukan sekadar
    potongan pertama yang memuat kata "mengadili" di mana saja dalam teks
    (yang sebelumnya bisa menangkap kutipan disclaimer atau bagian lain).
    """
    text_lc = text.lower()

    mengadili_pattern = re.compile(r'm\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i', re.IGNORECASE)
    matches = list(mengadili_pattern.finditer(text))

    if not matches:
        return extract_section(
            text,
            start_keywords=['memutuskan', 'amar putusan', 'mengabulkan', 'menolak', 'menetapkan'],
            end_keywords=['demikianlah', 'diputuskan dalam', 'putusan ini'],
            max_words=max_words,
        )

    # Ambil kemunculan terakhir kata "mengadili" karena biasanya itu adalah
    # diktum final (kemunculan sebelumnya bisa berupa kutipan putusan tingkat
    # bawah di dalam pertimbangan kasasi/PK).
    start_idx = matches[-1].end()
    candidate = text[start_idx:start_idx + 4000]

    end_keywords = ['demikianlah', 'diputuskan dalam', 'putusan ini diucapkan']
    end_idx = len(candidate)
    candidate_lc = candidate.lower()
    for kw in end_keywords:
        idx = candidate_lc.find(kw)
        if idx != -1:
            end_idx = min(end_idx, idx)

    section = candidate[:end_idx]
    words = section.split()[:max_words]
    result = ' '.join(words).strip()

    if not result:
        return extract_section(
            text,
            start_keywords=['memutuskan', 'amar putusan', 'mengabulkan', 'menolak', 'menetapkan'],
            end_keywords=['demikianlah', 'diputuskan dalam', 'putusan ini'],
            max_words=max_words,
        )

    return result


def extract_solution_label(amar_text: str) -> str:
    t = amar_text.lower()

    if 'mengabulkan' in t and 'sebagian' in t:
        return 'mengabulkan_sebagian'
    if 'mengabulkan' in t:
        return 'mengabulkan'
    if 'tidak dapat diterima' in t or 'niet ontvankelijk' in t:
        return 'tidak_dapat_diterima'
    if 'menolak' in t or 'ditolak' in t:
        return 'menolak'
    if 'membatalkan' in t:
        return 'membatalkan'
    if 'menetapkan' in t:
        return 'menetapkan'

    return 'lainnya'


def make_qa_pairs(no_perkara, pihak, pasal, tanggal):
    return ' // '.join([
        f'Q: Apa nomor perkara kasus ini? | A: {no_perkara}.',
        f'Q: Kapan tanggal putusan? | A: {tanggal}.',
        f'Q: Siapa pihak yang bersengketa? | A: {pihak}.',
        f'Q: Apa pasal atau rujukan hukum utama? | A: {pasal}.',
    ])


print('Fungsi ekstraksi metadata siap digunakan.')

Fungsi ekstraksi metadata siap digunakan.


## Validasi Domain Perkara

Sebelum representasi kasus dibangun, setiap dokumen diperiksa apakah benar-benar berisi perkara Perdata Waris. Validasi ini diperlukan karena hasil pengecekan manual menemukan ada dokumen yang lolos proses scraping/penamaan domain tetapi isinya bukan perkara waris (misalnya wanprestasi atas purchase order yang tidak ada hubungannya dengan warisan).

Dokumen ditandai sebagai relevan apabila mengandung kata kunci inti domain waris (`waris`, `ahli waris`, `harta peninggalan`, `legitieme portie`, `faraid`, `wasiat`, `pewaris`, dan sejenisnya) dengan frekuensi yang memadai relatif terhadap panjang dokumen. Dokumen yang tidak relevan dikeluarkan dari case base final dan dicatat di `logs/domain_validation.csv` agar dapat diganti dengan dokumen yang sesuai.

In [25]:
WARIS_KEYWORDS = [
    'ahli waris', 'harta waris', 'harta warisan', 'harta peninggalan',
    'pewaris', 'mewaris', 'waris mewaris', 'hukum waris', 'wasiat',
    'legitieme portie', 'faraid', 'warisan', 'kewarisan',
]

MIN_KEYWORD_HITS = 3
MIN_KEYWORD_DENSITY_PER_1000_WORDS = 1.0


def check_domain_relevance(text: str) -> dict:
    """
    Mengecek relevansi dokumen terhadap domain Perdata Waris.

    Dokumen dianggap relevan apabila jumlah kemunculan kata kunci waris
    memenuhi ambang batas minimum, baik secara absolut maupun secara
    kepadatan terhadap panjang dokumen (per 1000 kata). Dua ambang batas ini
    mencegah dokumen panjang yang hanya menyebut kata "waris" sekali secara
    sambil lalu tetap lolos sebagai relevan.
    """
    t = text.lower()
    word_count = max(len(t.split()), 1)

    keyword_hits = {}
    total_hits = 0
    for kw in WARIS_KEYWORDS:
        count = t.count(kw)
        if count > 0:
            keyword_hits[kw] = count
            total_hits += count

    density_per_1000 = (total_hits / word_count) * 1000

    is_relevant = (
        total_hits >= MIN_KEYWORD_HITS
        and density_per_1000 >= MIN_KEYWORD_DENSITY_PER_1000_WORDS
    )

    return {
        'total_keyword_hits': total_hits,
        'keyword_density_per_1000_words': round(density_per_1000, 3),
        'keyword_hits_detail': keyword_hits,
        'is_relevant_domain': is_relevant,
    }


print('Fungsi check_domain_relevance() siap digunakan.')
print(f'Ambang batas: minimal {MIN_KEYWORD_HITS} kemunculan kata kunci waris,')
print(f'dan kepadatan minimal {MIN_KEYWORD_DENSITY_PER_1000_WORDS} per 1000 kata.')

Fungsi check_domain_relevance() siap digunakan.
Ambang batas: minimal 3 kemunculan kata kunci waris,
dan kepadatan minimal 1.0 per 1000 kata.


In [26]:
def represent_case(case_id: str, text: str) -> dict:
    text = strip_extra_noise(text)

    no_perkara = extract_no_perkara(text)
    tanggal = extract_tanggal(text)
    pasal = extract_pasal(text)
    pihak = extract_pihak(text)

    ringkasan_fakta = extract_section(
        text,
        start_keywords=['duduk perkara', 'tentang duduk perkara', 'menimbang bahwa', 'posita', 'gugatan'],
        end_keywords=['pertimbangan hukum', 'mengadili', 'mengingat', 'menimbang, bahwa'],
        max_words=450,
    )

    argumen_hukum_utama = extract_amar_putusan(text, max_words=350)

    length_jumlah_kata = len(text.split())

    keywords = [
        'waris', 'ahli waris', 'harta', 'harta waris', 'tanah', 'gugatan',
        'kasasi', 'penggugat', 'tergugat', 'pemohon', 'termohon',
        'kompilasi hukum islam', 'kuhperdata'
    ]
    bow_counts = {kw.replace(' ', '_'): text.lower().count(kw) for kw in keywords}
    bag_of_words = '; '.join([f'{k}:{v}' for k, v in bow_counts.items()])

    qa_pairs = make_qa_pairs(no_perkara, pihak, pasal, tanggal)
    solution_label = extract_solution_label(argumen_hukum_utama)
    domain_check = check_domain_relevance(text)

    return {
        'case_id': case_id,
        'no_perkara': no_perkara,
        'tanggal': tanggal,
        'jenis_perkara': DOMAIN_PERKARA,
        'ringkasan_fakta': ringkasan_fakta,
        'argumen_hukum_utama': argumen_hukum_utama,
        'pasal': pasal,
        'pihak': pihak,
        'solution_label': solution_label,
        'length_jumlah_kata': length_jumlah_kata,
        'bag_of_words': bag_of_words,
        'qa_pairs': qa_pairs,
        'text_full': text,
        'domain_keyword_hits': domain_check['total_keyword_hits'],
        'domain_keyword_density': domain_check['keyword_density_per_1000_words'],
        'is_relevant_domain': domain_check['is_relevant_domain'],
    }


print('Fungsi represent_case() siap digunakan.')

Fungsi represent_case() siap digunakan.


In [27]:
text_files = sorted(RAW_TXT_DIR.glob('case_*.txt'))
print(f'File teks ditemukan: {len(text_files)}')

if len(text_files) < TARGET_DOCS:
    print(f'PERINGATAN: minimal {TARGET_DOCS} dokumen. Saat ini baru {len(text_files)} dokumen.')

rows = []
for path in text_files:
    text = path.read_text(encoding='utf-8', errors='ignore')
    rows.append(represent_case(path.stem, text))

df_all = pd.DataFrame(rows)
print(f'Berhasil membuat representasi untuk {len(df_all)} kasus (sebelum filter domain).')

if not df_all.empty:
    display(df_all[['case_id', 'no_perkara', 'tanggal', 'jenis_perkara', 'pasal', 'pihak', 'solution_label', 'length_jumlah_kata']].head())
    print('\nJumlah nilai TIDAK DITEMUKAN per kolom metadata:')
    for col in ['no_perkara', 'tanggal', 'pasal', 'pihak']:
        print(col, ':', int((df_all[col] == 'TIDAK DITEMUKAN').sum()))

File teks ditemukan: 50
Berhasil membuat representasi untuk 50 kasus (sebelum filter domain).


,case_id,no_perkara,tanggal,jenis_perkara,pasal,pihak,solution_label,length_jumlah_kata
0,case_001,62 PK/PDT/2025,19 juli 2024,Perdata Waris,"HUKUM ADAT, HUKUM WARIS, HUKUM WARIS / HIBAH",AHLI WARIS / PARA PEMOHON / PARA PENGGUGAT / P...,menolak,2938
1,case_002,82 K/PDT/2026,3 juni 2024,Perdata Waris,"HUKUM ADAT, HUKUM WARIS, KOMPILASI HUKUM ISLAM...",AHLI WARIS / PARA TERGUGAT / PARA TERMOHON / P...,mengabulkan,2795
2,case_003,169 K/PDT/2026,11 agustus 2025,Perdata Waris,"HUKUM ADAT, HUKUM WARIS, HUKUM WARIS / BOEDEL",AHLI WARIS / PARA PEMOHON / PARA TERGUGAT / PE...,menolak,2564
3,case_004,286 K/PDT/2026,2 september 2024,Perdata Waris,"HUKUM ADAT, HUKUM WARIS, HUKUM WARIS / WASIAT",AHLI WARIS / PARA PEMOHON / PARA PENGGUGAT / P...,menolak,3308
4,case_005,311 K/PDT/2026,14 juni 2025,Perdata Waris,"PASAL 66 KUHPERDATA, PASAL 832 KUHPERDATA, PAS...",AHLI WARIS / PEMOHON / PEMOHON KASASI / PENGGU...,menolak,2432



Jumlah nilai TIDAK DITEMUKAN per kolom metadata:
no_perkara : 0
tanggal : 0
pasal : 0
pihak : 0


## Filter Dokumen Berdasarkan Validasi Domain

Dokumen yang tidak memenuhi ambang batas relevansi domain waris dikeluarkan dari case base final. Daftar lengkap (baik yang relevan maupun tidak) disimpan di `logs/domain_validation.csv` agar dapat diaudit, dan dokumen yang ditandai tidak relevan sebaiknya diganti dengan dokumen waris lain dari Direktori MA RI sebelum pengumpulan tugas.

In [28]:
domain_log = df_all[[
    'case_id', 'no_perkara', 'domain_keyword_hits', 'domain_keyword_density', 'is_relevant_domain'
]].copy()

domain_log_path = LOG_DIR / 'domain_validation.csv'
domain_log.to_csv(domain_log_path, index=False)

n_relevant = int(df_all['is_relevant_domain'].sum())
n_excluded = len(df_all) - n_relevant

print(f'Total dokumen diproses     : {len(df_all)}')
print(f'Dokumen relevan (waris)    : {n_relevant}')
print(f'Dokumen dikeluarkan        : {n_excluded}')
print(f'Log validasi disimpan di   : {domain_log_path}')

if n_excluded > 0:
    print('\nDaftar dokumen yang dikeluarkan karena dianggap bukan perkara waris:')
    excluded_preview = df_all.loc[~df_all['is_relevant_domain'], ['case_id', 'no_perkara', 'domain_keyword_hits', 'domain_keyword_density']]
    display(excluded_preview)
    print('\nPERINGATAN: ganti dokumen di atas dengan putusan perdata waris lain dari Direktori MA RI,')
    print('lalu jalankan ulang notebook 01_preprocessing dan 02_representation.')

if n_relevant < TARGET_DOCS:
    print(f'\nPERINGATAN: setelah filter domain, jumlah dokumen relevan ({n_relevant}) masih di bawah syarat minimal {TARGET_DOCS}.')
else:
    print(f'\nSyarat jumlah dokumen relevan minimal {TARGET_DOCS}: TERPENUHI ({n_relevant} dokumen).')

df_cases = df_all.loc[df_all['is_relevant_domain']].drop(
    columns=['domain_keyword_hits', 'domain_keyword_density', 'is_relevant_domain']
).reset_index(drop=True)

print(f'\nJumlah kasus pada case base final (setelah filter domain): {len(df_cases)}')

Total dokumen diproses     : 50
Dokumen relevan (waris)    : 48
Dokumen dikeluarkan        : 2
Log validasi disimpan di   : d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\logs\domain_validation.csv

Daftar dokumen yang dikeluarkan karena dianggap bukan perkara waris:


,case_id,no_perkara,domain_keyword_hits,domain_keyword_density
10,case_011,727 PK/PDT/2025,2,0.972
15,case_016,1042 PK/PDT/2024,0,0.000



PERINGATAN: ganti dokumen di atas dengan putusan perdata waris lain dari Direktori MA RI,
lalu jalankan ulang notebook 01_preprocessing dan 02_representation.

Syarat jumlah dokumen relevan minimal 30: TERPENUHI (48 dokumen).

Jumlah kasus pada case base final (setelah filter domain): 48


In [29]:
csv_path = PROCESSED_DIR / 'cases.csv'
json_path = PROCESSED_DIR / 'cases.json'

df_cases.to_csv(csv_path, index=False, encoding='utf-8')
df_cases.to_json(json_path, orient='records', indent=2, force_ascii=False)

print('Output tahap 2 selesai.')
print('CSV :', csv_path)
print('JSON:', json_path)
print(f'Jumlah kasus pada case base final: {len(df_cases)}')

print('\nDistribusi solution_label pada case base final:')
print(df_cases['solution_label'].value_counts().to_string())

Output tahap 2 selesai.
CSV : d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\processed\cases.csv
JSON: d:\Semester 6\Penalaran Komputer\Tugas 3 -PK\data\processed\cases.json
Jumlah kasus pada case base final: 48

Distribusi solution_label pada case base final:
solution_label
menolak                 36
mengabulkan_sebagian     8
mengabulkan              3
tidak_dapat_diterima     1


## Kesimpulan Tahap 2

Tahap 2 menghasilkan representasi terstruktur dari case base yang sudah divalidasi relevansi domainnya.

Output yang dihasilkan:

1. `data/processed/cases.csv` dan `data/processed/cases.json` berisi kasus yang lolos validasi domain Perdata Waris.
2. `logs/domain_validation.csv` mencatat hasil pengecekan relevansi domain untuk seluruh dokumen, termasuk yang dikeluarkan.
3. Data siap digunakan pada tahap berikutnya, yaitu Case Retrieval.